# Cuaderno de Práctica: Funciones en Python para Data Science
Este cuaderno contiene ejercicios diseñados para aplicar los conceptos teóricos de la Sesión 4. 
Intenta resolver cada celda de código por tu cuenta antes de mirar las soluciones.

## Ejercicio 1: Herencia y Polimorfismo en Preprocesamiento 🧬
**Contexto:** En el documento vimos una clase `Dataset` que imputaba valores nulos usando el promedio (media). En series de tiempo, a menudo es mejor usar el último valor válido conocido (técnica "forward fill").
**Tarea:** 1. Crea una clase `TimeSeriesDataset` que herede de la clase `Dataset` original.
2. Sobrescribe (override) el método `transform(cols)` para que, en lugar de usar los promedios aprendidos en `fit()`, reemplace cualquier valor nulo con el **último valor válido que haya visto en esa misma columna** al iterar sobre las filas. Si el primer valor es nulo, déjalo como nulo.

In [13]:
#TENEMOS LA CLASE Dataset YA VISTA EN LA CLASE:
class Dataset:
    def __init__(self, rows):
        self.rows = rows
        self.promedios = {}

    def _to_float_safe(self, x):
        try:
            if x is None or x == "" or x == "NA":
                return None
            if type(x) == str:
                x = x.strip()
                if x == "":
                    return None
            return float(x)
        except:
            return None

    def fit(self, cols):
        # calcula promedios por columna
        for c in cols:
            total = 0
            n = 0
            for r in self.rows:
                v = self._to_float_safe(r.get(c))
                if v is None:
                    continue
                total += v
                n += 1
            self.promedios[c] = (total / n) if n > 0 else None
        return self

    def transform(self, cols):
        # retorna NUEVAS filas (no muta el dataset original)
        nuevas = []
        for r in self.rows:
            nuevo = {}
            for k, v in r.items():
                nuevo[k] = v

            for c in cols:
                v = self._to_float_safe(nuevo.get(c))
                if v is None:
                    # imputación simple por promedio
                    nuevo[c] = self.promedios.get(c)
                else:
                    nuevo[c] = v

            nuevas.append(nuevo)

        return Dataset(nuevas)


In [14]:
class TimeSeriesDataset(Dataset):
    def transform(self, cols):
        """
        Sobrescribe el método transform para usar 'forward fill'.
        Reemplaza NaN con el último valor válido encontrado en la misma columna.
        Si el primer valor es NaN, se deja como NaN.
        """
        # Copiamos las filas para no modificar la lista original directamente si no es deseado
        # Asumiendo que self.rows es una lista de diccionarios
        for col in cols:
            last_valid_value = None
            
            # Iteramos por el índice para poder acceder y modificar en el lugar
            for i in range(len(self.rows)):
                valor_actual = self.rows[i].get(col)
                
                # Verificamos si es nulo (None, np.nan, o string "NaN" dependiendo de tus datos)
                # Asumiremos None o float('nan')
                if valor_actual is None or (isinstance(valor_actual, float) and str(valor_actual) == 'nan'):
                    # Si es nulo, reemplazamos con el último valor válido VISTO
                    self.rows[i] [col] = last_valid_value
                else:
                    # Si no es nulo, actualizamos nuestro registro de "último válido"
                    last_valid_value = valor_actual
                    
        return self.rows


In [16]:
# --- Ejemplos de usos y pruebas ---
#PRIMER EJEMPLO:
# 1. Definir datos de ejemplo (simulando lo que vendría de tu Dataset original)
datos_ejemplo = [
    {'id': 1, 'gasto': 100},
    {'id': 2, 'gasto': None},      # NaN
    {'id': 3, 'gasto': None},      # NaN
    {'id': 4, 'gasto': 200},       # Nuevo valor válido
    {'id': 5, 'gasto': None},      # NaN
    {'id': 6, 'gasto': None}       # NaN
]

# 2. Instanciar y ejecutar
# Nota: Asumimos que la clase base Dataset no hace nada en __init__ que rompa esto
mi_dataset = TimeSeriesDataset(Dataset)
mi_dataset.rows = datos_ejemplo

# Llamamos al método transform
resultado = mi_dataset.transform(['gasto'])

# Imprimimos el resultado
for fila in resultado:
    print(fila)

{'id': 1, 'gasto': 100}
{'id': 2, 'gasto': 100}
{'id': 3, 'gasto': 100}
{'id': 4, 'gasto': 200}
{'id': 5, 'gasto': 200}
{'id': 6, 'gasto': 200}


In [18]:
#SEGUNDO EJEMPLO: Lo aplicaremos para el dataset de la clase
rows = [
    {"id": "C1", "gasto": "120", "visitas": "3"},
    {"id": "C2", "gasto": "NA",  "visitas": "1"},
    {"id": "C3", "gasto": "80",  "visitas": None},
    {"id": "C4", "gasto": "200", "visitas": "5"},
]

ds = TimeSeriesDataset(Dataset)
ds.rows = rows

ds2 = ds.transform(['gasto','visitas'])

for fila in ds2:
    print(fila)

{'id': 'C1', 'gasto': '120', 'visitas': '3'}
{'id': 'C2', 'gasto': 'NA', 'visitas': '1'}
{'id': 'C3', 'gasto': '80', 'visitas': '1'}
{'id': 'C4', 'gasto': '200', 'visitas': '5'}


## Ejercicio 2: Composición para crear un `Pipeline` de Datos ⚙️
**Contexto:** En Data Science, un "Pipeline" es una cadena de pasos secuenciales. Un objeto `Pipeline` *tiene* varios pasos (Composición).
**Tarea:**
1. Crea una clase `PasoNormalizacion` y otra clase `PasoFiltro`. Ambas deben tener un método `ejecutar(datos)` que reciba una lista de números y devuelva una lista modificada (aplica la lógica matemática que prefieras a cada una).
2. Crea una clase `Pipeline`. Su `__init__` debe recibir una lista de objetos "paso".
3. Crea un método `run(datos)` en `Pipeline` que tome los datos iniciales y los pase secuencialmente por el método `ejecutar()` de cada paso guardado en la lista, devolviendo el resultado final.

In [ ]:
class PasoNormalizacion:
    def __init__(self):

    def ejecutar(self,datos):
        
        
class PasoFiltro():
    def __init__(self):

    def ejecutar(self,datos):

    

class Pipeline:
    def __init__(self,paso):
        self.paso = paso

    def run(self,datos):
        paso1 = PasoFiltro(datos)
        return datos 
        
    

## Ejercicio 3: Encapsulación estricta con `@property` en Modelos ML 🛡️
**Contexto:** Los hiperparámetros de los modelos de Machine Learning deben estar estrictamente validados para evitar errores en entrenamiento.
**Tarea:** 1. Crea una clase `ConfiguracionArbol` con atributos como `profundidad_maxima` y `tasa_aprendizaje` (learning rate).
2. Utiliza los decoradores `@property` y `@atributo.setter` (vistos en la sección de encapsulación ligera) para garantizar que:
   - La `profundidad_maxima` sea un número entero estrictamente mayor a 0.
   - La `tasa_aprendizaje` sea un número flotante entre 0.0 y 1.0.
3. Lanza un `ValueError` con un mensaje claro si el usuario intenta asignar valores fuera de estos rangos.

In [28]:
class ConfiguracionArbol:
    def __init__(self,profundidad_maxima,tasa_aprendizaje):
        self.profundidad_maxima = profundidad_maxima
        self.tasa_aprendizaje = tasa_aprendizaje

    @property
    def profundidad_maxima(self):
        return self._profundidad_maxima

    @profundidad_maxima.setter
    def profundidad_maxima(self, valor):
        if not(isinstance(valor, int) and valor > 0):
            raise ValueError("La profundidad debe ser un entero positivo.")
        self._profundidad_maxima = valor

    @property
    def tasa_aprendizaje(self):
        return self._tasa_aprendizaje

    @tasa_aprendizaje.setter
    def tasa_aprendizaje(self, valor):
        if not (isinstance(valor,float) and 0 < valor < 1):
            raise ValueError("La tasa de aprendizaje debe estar entre 0 y 1.")
        self._tasa_aprendizaje = valor

In [31]:
xd = ConfiguracionArbol(3 , 0.5)
print(xd.profundidad_maxima)
print(xd.tasa_aprendizaje)

3
0.5


## Ejercicio 4: Métodos Mágicos Avanzados (`__add__` y `__sub__`) 🧮
**Contexto:** En el documento aprendiste a usar `__add__` con fracciones. Vamos a aplicarlo a vectores de características (Feature Vectors).
**Tarea:**
1. Crea una clase `FeatureVector` que se inicialice con una lista de valores numéricos.
2. Implementa el método mágico `__add__` para sumar dos objetos `FeatureVector` posición por posición (asumiendo que tienen la misma longitud).
3. Implementa el método mágico `__sub__` para restarlos.
4. Implementa `__str__` para que al hacer `print(vector)` muestre un texto como: `[FeatureVector -> values: (1, 5, 3)]`.

In [49]:
class FeatureVector:
    def __init__(self,lista):
        if len(lista) == 0:
            raise ValueError("Escriba lista no vacías")
        self.lista = lista
    def __addv__(self,otra):
        if not(len(self.lista) == len(otra)):
            raise ValueError("Otorgue lista con el mismo tamaño que la que tiene")
        nuevo = []
        for i in range(len(self.lista)):
            nuevo.append(self.lista[i].__add__(otra[i]))
        return nuevo

vector=FeatureVector([0,1,2,3])
print(vector.__addv__([3,4,5,6]))

[3, 5, 7, 9]


## Ejercicio 5: Ampliando la `ConfusionMatrix` con F1-Score 🎯
**Contexto:** Tienes la clase `ConfusionMatrix` (DS 4 del documento) que calcula Accuracy, Precision y Recall.
**Tarea:**
1. Añade un nuevo método llamado `f1_score(self)`. 
2. El F1-Score se calcula usando Precision y Recall con la fórmula: `2 * (Precision * Recall) / (Precision + Recall)`.
3. Tu método `f1_score` **no debe** calcular directamente usando `self.tp`, `self.fp`, etc., sino que debe **llamar internamente a los métodos** `self.precision()` y `self.recall()`. Asegúrate de manejar correctamente el caso en el que el denominador sea 0 para no romper el programa.